In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(".."))

from src.data import get_xy
from src.models import evaluate_model, print_results, print_test_metrics, get_permutation_importance, print_coefficients
from src import visalization as viz

In [ ]:
df_train = pd.read_excel("training_barcelona_shots.xlsx", header=0)
df_test = pd.read_excel("testing_barcelona_shots.xlsx", header=0)

shot_features = [
    "distance_d",
    "angle_d",
    "free_kick_flag",
    "penalty_flag",
    "technique_b",
    "n_def_1_5",
    "n_def_3_0",
    "dist_nearest_def",
    "gk_dist_to_shooter"
]

X_train, y_train = get_xy(df_train, shot_features)
X_test, y_test = get_xy(df_test, shot_features)

In [ ]:
model_xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 1, 5]
}

In [ ]:
grid_xgb, results_xgb = evaluate_model(model_xgb, param_grid_xgb, X_train, y_train)
print_results(results_xgb)

In [ ]:
final_model_xgb = XGBClassifier(**grid_xgb.best_params_, eval_metric="logloss", random_state=42)
final_model_xgb.fit(X_train, y_train)

y_pred_proba_xgb = final_model_xgb.predict_proba(X_test)[:, 1]

In [ ]:
df_test["predicted_xg"] = y_pred_proba_xgb

statsbomb_xg = df_test["statsbomb_xg"]
pred_xg_xgb = df_test["predicted_xg"]

total_pred_xg_xgb = pred_xg_xgb.sum()
total_statsbomb_xg = statsbomb_xg.sum()
total_goals = y_test.sum()

print(f"Total Predicted xG: {total_pred_xg_xgb:.2f}")
print(f"Total StatsBomb xG: {total_statsbomb_xg:.2f}")
print(f"Actual Goals: {total_goals}")

In [ ]:
print_test_metrics(y_test, pred_xg_xgb, statsbomb_xg)
correlation_xgb = np.corrcoef(statsbomb_xg, pred_xg_xgb)[0, 1]
mae_xgb = np.mean(np.abs(statsbomb_xg - pred_xg_xgb))
print(f"\nCorrelation (Model xG vs StatsBomb xG): {correlation_xgb:.3f}")
print(f"Mean Absolute Error: {mae_xgb:.3f}")

In [ ]:
viz.plot_correlation(statsbomb_xg, pred_xg_xgb)

In [ ]:
viz.plot_calibration(y_test, pred_xg_xgb, statsbomb_xg)

In [ ]:
week_xg_xgb = df_test.groupby('week').agg({
    'predicted_xg': 'sum',
    'statsbomb_xg': 'sum',
    'goal/no goal': 'sum'
}).reset_index()

viz.plot_weekly_xg(week_xg_xgb)

In [ ]:
viz.plot_roc_curve(y_test, pred_xg_xgb, statsbomb_xg)

In [ ]:
importances_df_xgb = get_permutation_importance(final_model_xgb, X_test, y_test)
viz.plot_feature_importances(importances_df_xgb, "XGBoost Classifier")

In [ ]:
scaler_lr = StandardScaler()

X_train_scaled_lr = scaler_lr.fit_transform(X_train)
X_test_scaled_lr = scaler_lr.transform(X_test)

In [ ]:
C_values = np.logspace(-6, 3, 50)

coefs = []

for C in C_values:
    lr = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000,
        C=C,
        random_state=42
    )
    lr.fit(X_train_scaled_lr, y_train)
    coefs.append(lr.coef_[0])

coefs = np.array(coefs)
viz.plot_l1_paths(C_values, coefs, X_train.columns)

In [ ]:
model_lr = LogisticRegression(
    max_iter=1000, 
    random_state=42
)

param_grid_lr = {
    "penalty": ["l1"],
    "C": [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1],
    "solver": ["liblinear"]
}

In [ ]:
grid_lr, results_lr = evaluate_model(model_lr, param_grid_lr, X_train_scaled_lr, y_train)
print_results(results_lr)

In [ ]:
best_lr = grid_lr.best_estimator_
coef = best_lr.coef_[0]
print_coefficients(X_train.columns, coef)